# 🎸 Genre-Analyse

Genre-Informationen sind nicht in den Spotify-Exportdaten enthalten.
Dieses Notebook nutzt die **Spotify Web API** um Artist-Genres nachzuschlagen.

### Setup
1. Erstelle eine App auf https://developer.spotify.com/dashboard
2. Erstelle eine `.env` Datei im Projektroot:
   ```
   SPOTIPY_CLIENT_ID=dein_client_id
   SPOTIPY_CLIENT_SECRET=dein_client_secret
   ```
3. Führe dieses Notebook aus

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from config import *
from data_loader import load_data, get_music, ensure_results_dir

plt.style.use(PLOT_STYLE)
plt.rcParams['figure.dpi'] = PLOT_DPI
plt.rcParams['savefig.dpi'] = PLOT_DPI
plt.rcParams['savefig.bbox'] = 'tight'

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')
    import spotipy
    from spotipy.oauth2 import SpotifyClientCredentials
    sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials())
    API_AVAILABLE = True
    print('✅ Spotify API verbunden')
except Exception as e:
    API_AVAILABLE = False
    print(f'⚠️ Spotify API nicht verfügbar: {e}')
    print('Bitte .env Datei mit SPOTIPY_CLIENT_ID und SPOTIPY_CLIENT_SECRET erstellen.')

In [ ]:
df = load_data()
music = get_music(df)

## Genre-Cache laden oder erstellen

Die API wird nur für neue Artists aufgerufen. Ergebnisse werden in `.genre_cache.json` gespeichert.

In [ ]:
cache_file = SPOTIFY_GENRE_CACHE_FILE

if cache_file.exists():
    with open(cache_file, 'r', encoding='utf-8') as f:
        genre_cache = json.load(f)
    print(f'Cache geladen: {len(genre_cache)} Artists')
else:
    genre_cache = {}
    print('Kein Cache vorhanden, starte frisch.')

In [ ]:
if not API_AVAILABLE:
    print('⛔ Überspringe Genre-Lookup – API nicht verfügbar.')
else:
    unique_artists = music['artist'].dropna().unique()
    new_artists = [a for a in unique_artists if a not in genre_cache]
    print(f'{len(new_artists)} neue Artists zum Nachschlagen (von {len(unique_artists)} gesamt)')

    for i, artist_name in enumerate(new_artists):
        try:
            results = sp.search(q=f'artist:{artist_name}', type='artist', limit=1)
            items = results['artists']['items']
            if items:
                genre_cache[artist_name] = items[0].get('genres', [])
            else:
                genre_cache[artist_name] = []
        except Exception as e:
            genre_cache[artist_name] = []

        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(new_artists)} abgefragt...')
            # Zwischenspeichern
            with open(cache_file, 'w', encoding='utf-8') as f:
                json.dump(genre_cache, f, ensure_ascii=False, indent=2)

    # Final speichern
    with open(cache_file, 'w', encoding='utf-8') as f:
        json.dump(genre_cache, f, ensure_ascii=False, indent=2)
    print(f'\n✅ Genre-Cache gespeichert: {len(genre_cache)} Artists')

## Genres den Streams zuordnen

In [ ]:
if not genre_cache:
    print('⛔ Kein Genre-Cache vorhanden. Bitte zuerst API-Lookup durchführen.')
else:
    music_g = music.copy()
    music_g['genres'] = music_g['artist'].map(lambda a: genre_cache.get(a, []))

    # Genres "exploden" – jeder Stream kann mehrere Genres haben
    exploded = music_g.explode('genres').dropna(subset=['genres'])
    exploded = exploded[exploded['genres'] != '']
    print(f'{len(exploded)} Stream-Genre-Zuordnungen erstellt')

## Top Genres – Gesamte Hörzeit

In [ ]:
if genre_cache:
    top_genres = (exploded.groupby('genres')['hours_played'].sum()
                 .sort_values(ascending=False).head(25).reset_index())

    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_TALL)
    bars = ax.barh(range(len(top_genres)), top_genres['hours_played'], color=COLOR_PRIMARY)
    ax.set_yticks(range(len(top_genres)))
    ax.set_yticklabels(top_genres['genres'])
    ax.invert_yaxis()
    for bar, h in zip(bars, top_genres['hours_played']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{h:,.1f}h', va='center', fontsize=9)
    ax.set_xlabel('Stunden')
    ax.set_title('🎸 Top 25 Genres (Gesamte Hörzeit)', fontsize=16, fontweight='bold')
    plt.tight_layout()

    out = ensure_results_dir('overview')
    fig.savefig(out / 'top_genres_alltime.png')
    plt.show()

## Genre-Verteilung pro Jahr

In [ ]:
if genre_cache:
    # Top 10 Genres global für Konsistenz
    top10_genres = top_genres['genres'].head(10).tolist()
    years = sorted(exploded['year'].unique())

    for year in years:
        year_data = exploded[exploded['year'] == year]
        yg = (year_data.groupby('genres')['hours_played'].sum()
              .sort_values(ascending=False).head(15).reset_index())

        fig, ax = plt.subplots(figsize=(12, 8))
        ax.barh(range(len(yg)), yg['hours_played'], color=COLOR_PALETTE[:len(yg)])
        ax.set_yticks(range(len(yg)))
        ax.set_yticklabels(yg['genres'])
        ax.invert_yaxis()
        for i, h in enumerate(yg['hours_played']):
            ax.text(h + 0.2, i, f'{h:.1f}h', va='center', fontsize=9)
        ax.set_xlabel('Stunden')
        ax.set_title(f'🎸 Top Genres {year}', fontsize=14, fontweight='bold')
        plt.tight_layout()

        year_out = ensure_results_dir(year)
        fig.savefig(year_out / 'top_genres.png')
        plt.show()
        print(f'Gespeichert: {year_out / "top_genres.png"}')

## Genre-Trend über die Jahre

In [ ]:
if genre_cache:
    genre_year = (exploded[exploded['genres'].isin(top10_genres)]
                  .groupby(['year', 'genres'])['hours_played'].sum()
                  .reset_index()
                  .pivot(index='year', columns='genres', values='hours_played')
                  .fillna(0))

    fig, ax = plt.subplots(figsize=(16, 8))
    genre_year.plot(kind='area', stacked=True, ax=ax,
                    color=COLOR_PALETTE[:len(top10_genres)], alpha=0.8)
    ax.set_xlabel('Jahr')
    ax.set_ylabel('Stunden')
    ax.set_title('🎸 Genre-Entwicklung über die Jahre (Top 10)', fontsize=16, fontweight='bold')
    ax.legend(title='Genre', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()

    out = ensure_results_dir('overview')
    fig.savefig(out / 'genre_trend_over_years.png')
    plt.show()